In [ ]:
# Supply Chain Risk Prediction Using M5 Walmart Sales Data
# Main Implementation Notebook

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("🚀 Supply Chain Risk Prediction Project")
print("=" * 50)

# ============================================================================
# SECTION 1: DATA LOADING AND INITIAL EXPLORATION
# ============================================================================

def load_m5_data(data_path="./data/"):
    """
    Load M5 competition datasets

    Download from: https://www.kaggle.com/c/m5-forecasting-accuracy/data
    """
    print("📊 Loading M5 datasets...")

    try:
        # Load main datasets
        sales_df = pd.read_csv(f"{data_path}sales_train_evaluation.csv")
        calendar_df = pd.read_csv(f"{data_path}calendar.csv")
        prices_df = pd.read_csv(f"{data_path}sell_prices.csv")

        print(f"✅ Sales data loaded: {sales_df.shape}")
        print(f"✅ Calendar data loaded: {calendar_df.shape}")
        print(f"✅ Price data loaded: {prices_df.shape}")

        return sales_df, calendar_df, prices_df

    except FileNotFoundError as e:
        print("❌ Data files not found!")
        print("Please download M5 data from Kaggle and place in ./data/ directory")
        print("Required files: sales_train_evaluation.csv, calendar.csv, sell_prices.csv")
        return None, None, None

def explore_data_structure(sales_df, calendar_df, prices_df):
    """Initial data exploration"""

    print("\n🔍 DATA EXPLORATION")
    print("=" * 30)

    # Sales data structure
    print("📈 Sales Data Structure:")
    print(f"Shape: {sales_df.shape}")
    print(f"Stores: {sales_df['store_id'].nunique()} unique stores")
    print(f"Categories: {sales_df['cat_id'].nunique()} categories")
    print(f"Departments: {sales_df['dept_id'].nunique()} departments")
    print(f"Items: {sales_df['item_id'].nunique()} unique items")

    # Calendar data
    print(f"\n📅 Calendar Data:")
    print(f"Date range: {calendar_df['date'].min()} to {calendar_df['date'].max()}")
    print(f"Total days: {len(calendar_df)}")
    print(f"Events: {calendar_df['event_name_1'].nunique()} unique events")

    # Price data
    print(f"\n💰 Price Data:")
    print(f"Price entries: {len(prices_df)}")
    print(f"Price range: ${prices_df['sell_price'].min():.2f} - ${prices_df['sell_price'].max():.2f}")

    # Sample data preview
    print(f"\n📋 Sample Sales Data:")
    print(sales_df.head())

    return True

# Load data
sales_df, calendar_df, prices_df = load_m5_data()

if sales_df is not None:
    explore_data_structure(sales_df, calendar_df, prices_df)

# ============================================================================
# SECTION 2: DATA PREPROCESSING AND TRANSFORMATION
# ============================================================================

def transform_sales_data(sales_df, calendar_df):
    """
    Transform wide sales data to long format and merge with calendar
    """
    print("\n🔄 Transforming sales data to time series format...")

    # Get day columns (d_1, d_2, ..., d_1941)
    day_cols = [col for col in sales_df.columns if col.startswith('d_')]

    # Melt sales data from wide to long format
    id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
    sales_long = pd.melt(
        sales_df,
        id_vars=id_cols,
        value_vars=day_cols,
        var_name='d',
        value_name='sales'
    )

    # Merge with calendar to get actual dates
    sales_ts = sales_long.merge(calendar_df[['d', 'date', 'wm_yr_wk']], on='d', how='left')
    sales_ts['date'] = pd.to_datetime(sales_ts['date'])

    # Sort by item and date
    sales_ts = sales_ts.sort_values(['id', 'date']).reset_index(drop=True)

    print(f"✅ Transformed data shape: {sales_ts.shape}")
    print(f"✅ Date range: {sales_ts['date'].min()} to {sales_ts['date'].max()}")

    return sales_ts

def create_sample_dataset(sales_ts, n_items=100, start_date='2014-01-01', end_date='2016-06-01'):
    """
    Create a manageable sample for development and testing
    """
    print(f"\n🎯 Creating sample dataset...")

    # Filter date range
    mask = (sales_ts['date'] >= start_date) & (sales_ts['date'] <= end_date)
    sample_data = sales_ts[mask].copy()

    # Select top N items by total sales volume
    item_sales = sample_data.groupby('id')['sales'].sum().sort_values(ascending=False)
    top_items = item_sales.head(n_items).index.tolist()

    sample_data = sample_data[sample_data['id'].isin(top_items)].copy()

    print(f"✅ Sample dataset created:")
    print(f"   Items: {sample_data['id'].nunique()}")
    print(f"   Stores: {sample_data['store_id'].nunique()}")
    print(f"   Date range: {sample_data['date'].min()} to {sample_data['date'].max()}")
    print(f"   Total records: {len(sample_data)}")

    return sample_data

# Transform data
if sales_df is not None:
    sales_ts = transform_sales_data(sales_df, calendar_df)
    sample_data = create_sample_dataset(sales_ts, n_items=50)

# ============================================================================
# SECTION 3: FEATURE ENGINEERING
# ============================================================================

class SupplyChainFeatureEngineer:
    """Feature engineering class for supply chain risk prediction"""

    def __init__(self, sales_data, calendar_data, price_data=None):
        self.sales_data = sales_data.copy()
        self.calendar_data = calendar_data.copy()
        self.price_data = price_data.copy() if price_data is not None else None

    def create_rolling_features(self, windows=[7, 14, 28]):
        """Create rolling statistical features for anomaly detection"""
        print("📊 Creating rolling statistical features...")

        features_list = []

        for item_id in self.sales_data['id'].unique():
            item_data = self.sales_data[self.sales_data['id'] == item_id].copy()
            item_data = item_data.sort_values('date')

            item_features = item_data[['id', 'date', 'sales']].copy()

            for window in windows:
                # Rolling statistics
                item_features[f'rolling_mean_{window}'] = item_data['sales'].rolling(window, min_periods=1).mean()
                item_features[f'rolling_std_{window}'] = item_data['sales'].rolling(window, min_periods=1).std()
                item_features[f'rolling_median_{window}'] = item_data['sales'].rolling(window, min_periods=1).median()
                item_features[f'rolling_max_{window}'] = item_data['sales'].rolling(window, min_periods=1).max()
                item_features[f'rolling_min_{window}'] = item_data['sales'].rolling(window, min_periods=1).min()

                # Coefficient of variation (volatility measure)
                item_features[f'rolling_cv_{window}'] = (
                    item_features[f'rolling_std_{window}'] /
                    (item_features[f'rolling_mean_{window}'] + 1e-6)
                )

                # Z-score for anomaly detection
                item_features[f'zscore_{window}'] = (
                    (item_data['sales'] - item_features[f'rolling_mean_{window}']) /
                    (item_features[f'rolling_std_{window}'] + 1e-6)
                )

                # Trend features
                item_features[f'trend_{window}'] = (
                    item_features[f'rolling_mean_{window}'] -
                    item_features[f'rolling_mean_{window}'].shift(window)
                )

            features_list.append(item_features)

        rolling_features = pd.concat(features_list, ignore_index=True)
        print(f"✅ Rolling features created for {len(features_list)} items")

        return rolling_features

    def create_calendar_features(self):
        """Extract calendar and event features"""
        print("📅 Creating calendar features...")

        cal_features = self.calendar_data.copy()
        cal_features['date'] = pd.to_datetime(cal_features['date'])

        # Time-based features
        cal_features['day_of_week'] = cal_features['date'].dt.dayofweek
        cal_features['day_of_month'] = cal_features['date'].dt.day
        cal_features['week_of_year'] = cal_features['date'].dt.isocalendar().week
        cal_features['month'] = cal_features['date'].dt.month
        cal_features['quarter'] = cal_features['date'].dt.quarter
        cal_features['is_weekend'] = cal_features['day_of_week'].isin([5, 6]).astype(int)
        cal_features['is_month_start'] = cal_features['date'].dt.is_month_start.astype(int)
        cal_features['is_month_end'] = cal_features['date'].dt.is_month_end.astype(int)

        # Event features
        cal_features['has_event'] = (~cal_features['event_name_1'].isna()).astype(int)
        cal_features['event_count'] = (
            (~cal_features['event_name_1'].isna()).astype(int) +
            (~cal_features['event_name_2'].isna()).astype(int)
        )

        # SNAP benefits (Supplemental Nutrition Assistance Program)
        snap_cols = ['snap_CA', 'snap_TX', 'snap_WI']
        cal_features['snap_total'] = cal_features[snap_cols].sum(axis=1)

        # Holiday proximity features
        holiday_mask = cal_features['event_type_1'] == 'Religious'
        cal_features['days_to_holiday'] = self._calculate_days_to_event(cal_features, holiday_mask)
        cal_features['days_from_holiday'] = self._calculate_days_from_event(cal_features, holiday_mask)

        print("✅ Calendar features created")
        return cal_features

    def _calculate_days_to_event(self, cal_data, event_mask):
        """Calculate days until next event"""
        days_to_event = []
        event_dates = cal_data[event_mask]['date'].tolist()

        for date in cal_data['date']:
            future_events = [ed for ed in event_dates if ed > date]
            if future_events:
                days_to_event.append((min(future_events) - date).days)
            else:
                days_to_event.append(365)  # Default large value

        return days_to_event

    def _calculate_days_from_event(self, cal_data, event_mask):
        """Calculate days since last event"""
        days_from_event = []
        event_dates = cal_data[event_mask]['date'].tolist()

        for date in cal_data['date']:
            past_events = [ed for ed in event_dates if ed < date]
            if past_events:
                days_from_event.append((date - max(past_events)).days)
            else:
                days_from_event.append(365)  # Default large value

        return days_from_event

    def create_price_features(self):
        """Create price-related features if price data is available"""
        if self.price_data is None:
            print("⚠️ No price data available")
            return None

        print("💰 Creating price features...")

        price_features = self.price_data.copy()

        # Sort by store, item, and week
        price_features = price_features.sort_values(['store_id', 'item_id', 'wm_yr_wk'])

        # Calculate price changes
        price_features['price_change'] = price_features.groupby(['store_id', 'item_id'])['sell_price'].pct_change()
        price_features['price_change_abs'] = price_features['price_change'].abs()

        # Price volatility
        price_features['price_volatility'] = price_features.groupby(['store_id', 'item_id'])['sell_price'].rolling(4).std().reset_index(0, drop=True)

        # Flag significant price changes
        price_features['significant_price_increase'] = (price_features['price_change'] > 0.1).astype(int)
        price_features['significant_price_decrease'] = (price_features['price_change'] < -0.1).astype(int)
        price_features['any_price_change'] = (price_features['price_change_abs'] > 0.05).astype(int)

        print("✅ Price features created")
        return price_features

# Create feature engineer instance and generate features
if 'sample_data' in locals():
    feature_engineer = SupplyChainFeatureEngineer(sample_data, calendar_df, prices_df)

    # Generate all features
    rolling_features = feature_engineer.create_rolling_features()
    calendar_features = feature_engineer.create_calendar_features()
    price_features = feature_engineer.create_price_features()

# ============================================================================
# SECTION 4: DISRUPTION DETECTION AND LABELING
# ============================================================================

class DisruptionDetector:
    """Class for detecting and labeling supply chain disruptions"""

    def __init__(self, zscore_threshold=-2.0, min_duration=2, min_impact=0.15):
        self.zscore_threshold = zscore_threshold
        self.min_duration = min_duration
        self.min_impact = min_impact

    def detect_disruptions(self, sales_features, calendar_features, price_features=None):
        """
        Detect supply chain disruptions based on sales anomalies

        Disruption criteria:
        1. Significant drop in sales (z-score below threshold)
        2. Not explained by price changes or major events
        3. Sustained for minimum duration
        4. Minimum impact threshold
        """
        print("🚨 Detecting supply chain disruptions...")

        disruption_results = []

        # Group by item for individual analysis
        for item_id in sales_features['id'].unique():
            item_data = sales_features[sales_features['id'] == item_id].copy()
            item_data = item_data.sort_values('date')

            # Merge with calendar features
            item_data = item_data.merge(
                calendar_features[['date', 'has_event', 'event_count', 'snap_total']],
                on='date',
                how='left'
            )

            # Merge with price features if available
            if price_features is not None:
                # Match on week and store/item
                item_info = item_data[['store_id', 'item_id', 'wm_yr_wk']].drop_duplicates()
                price_subset = price_features[
                    price_features['item_id'].isin(item_info['item_id']) &
                    price_features['store_id'].isin(item_info['store_id'])
                ][['store_id', 'item_id', 'wm_yr_wk', 'any_price_change']]

                item_data = item_data.merge(price_subset, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')
                item_data['any_price_change'] = item_data['any_price_change'].fillna(0)
            else:
                item_data['any_price_change'] = 0

            # Detect anomalies using multiple z-score windows
            anomaly_mask = (
                (item_data['zscore_7'] < self.zscore_threshold) |
                (item_data['zscore_14'] < self.zscore_threshold)
            )

            # Filter out event-driven and price-driven drops
            event_mask = item_data['has_event'] == 1
            price_change_mask = item_data['any_price_change'] == 1
            snap_mask = item_data['snap_total'] > 0  # SNAP benefit days might cause irregular patterns

            # True disruptions: anomalies not explained by external factors
            disruption_mask = anomaly_mask & ~event_mask & ~price_change_mask & ~snap_mask

            # Calculate impact magnitude
            impact_magnitude = np.abs(item_data['zscore_14'])
            significant_impact = impact_magnitude > (np.abs(self.zscore_threshold) * 0.75)

            # Combine all conditions
            final_disruption_mask = disruption_mask & significant_impact

            # Find consecutive disruption periods
            disruption_periods = self._find_consecutive_periods(
                final_disruption_mask,
                self.min_duration
            )

            # Create disruption labels
            item_disruptions = self._create_disruption_labels(
                item_data,
                disruption_periods,
                final_disruption_mask
            )

            disruption_results.append(item_disruptions)

        # Combine all results
        all_disruptions = pd.concat(disruption_results, ignore_index=True)

        # Summary statistics
        total_disruptions = all_disruptions['disruption_label'].sum()
        disruption_rate = total_disruptions / len(all_disruptions) * 100

        print(f"✅ Disruption detection completed:")
        print(f"   Total observations: {len(all_disruptions):,}")
        print(f"   Disruptions detected: {total_disruptions:,}")
        print(f"   Disruption rate: {disruption_rate:.2f}%")

        return all_disruptions

    def _find_consecutive_periods(self, mask, min_duration=2):
        """Find consecutive True periods in boolean mask"""
        periods = []
        start = None

        for i, val in enumerate(mask):
            if val and start is None:
                start = i
            elif not val and start is not None:
                if i - start >= min_duration:
                    periods.append((start, i-1))
                start = None

        # Handle case where disruption continues to end of data
        if start is not None and len(mask) - start >= min_duration:
            periods.append((start, len(mask)-1))

        return periods

    def _create_disruption_labels(self, item_data, disruption_periods, disruption_mask):
        """Create disruption labels and metadata"""

        item_result = item_data.copy()
        item_result['disruption_label'] = 0
        item_result['disruption_severity'] = 'None'
        item_result['disruption_duration'] = 0
        item_result['disruption_impact'] = 0.0

        # Label disruption periods
        for start_idx, end_idx in disruption_periods:
            duration = end_idx - start_idx + 1

            # Calculate impact as average z-score magnitude during disruption
            period_data = item_result.iloc[start_idx:end_idx+1]
            avg_impact = np.mean(np.abs(period_data['zscore_14']))

            # Classify severity
            severity = self._classify_disruption_severity(avg_impact, duration)

            # Update labels
            item_result.iloc[start_idx:end_idx+1, item_result.columns.get_loc('disruption_label')] = 1
            item_result.iloc[start_idx:end_idx+1, item_result.columns.get_loc('disruption_severity')] = severity
            item_result.iloc[start_idx:end_idx+1, item_result.columns.get_loc('disruption_duration')] = duration
            item_result.iloc[start_idx:end_idx+1, item_result.columns.get_loc('disruption_impact')] = avg_impact

        return item_result

    def _classify_disruption_severity(self, avg_impact, duration):
        """Classify disruption severity based on impact and duration"""

        if avg_impact > 4.0 or duration > 14:
            return 'Critical'
        elif avg_impact > 3.0 or duration > 7:
            return 'High'
        elif avg_impact > 2.5 or duration > 3:
            return 'Medium'
        else:
            return 'Low'

# Initialize disruption detector and detect disruptions
if 'rolling_features' in locals():
    detector = DisruptionDetector(zscore_threshold=-2.0, min_duration=2)
    disruption_data = detector.detect_disruptions(rolling_features, calendar_features, price_features)

# ============================================================================
# SECTION 5: VISUALIZATION AND ANALYSIS
# ============================================================================

def visualize_disruption_examples(disruption_data, n_examples=3):
    """Visualize examples of detected disruptions"""

    print("📊 Visualizing disruption examples...")

    # Find items with disruptions
    disrupted_items = disruption_data[disruption_data['disruption_label'] == 1]['id'].unique()

    if len(disrupted_items) == 0:
        print("❌ No disruptions found to visualize")
        return

    # Select random examples
    example_items = np.random.choice(disrupted_items, min(n_examples, len(disrupted_items)), replace=False)

    fig, axes = plt.subplots(len(example_items), 1, figsize=(15, 5*len(example_items)))
    if len(example_items) == 1:
        axes = [axes]

    for i, item_id in enumerate(example_items):
        item_data = disruption_data[disruption_data['id'] == item_id].copy()
        item_data = item_data.sort_values('date')

        ax = axes[i]

        # Plot sales data
        ax.plot(item_data['date'], item_data['sales'], 'b-', linewidth=2, label='Daily Sales', alpha=0.7)
        ax.plot(item_data['date'], item_data['rolling_mean_14'], 'g--', linewidth=1, label='14-day Rolling Mean')

        # Highlight disruptions
        disruption_periods = item_data[item_data['disruption_label'] == 1]
        if len(disruption_periods) > 0:
            ax.scatter(disruption_periods['date'], disruption_periods['sales'],
                      color='red', s=50, label='Disruption', alpha=0.8, zorder=5)

            # Add severity information
            for severity in disruption_periods['disruption_severity'].unique():
                severity_data = disruption_periods[disruption_periods['disruption_severity'] == severity]
                ax.scatter(severity_data['date'], severity_data['sales'],
                          label=f'{severity} Disruption', s=60, alpha=0.9)

        ax.set_title(f'Supply Chain Disruptions - {item_id}', fontsize=14, fontweight='bold')
        ax.set_xlabel('Date')
        ax.set_ylabel('Sales Units')
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def analyze_disruption_patterns(disruption_data):
    """Analyze patterns in detected disruptions"""

    print("\n📈 DISRUPTION PATTERN ANALYSIS")
    print("=" * 40)

    # Overall statistics
    total_obs = len(disruption_data)
    total_disruptions = disruption_data['disruption_label'].sum()
    disruption_rate = total_disruptions / total_obs * 100

    print(f"📊 Overall Statistics:")
    print(f"   Total observations: {total_obs:,}")
    print(f"   Disruptions detected: {total_disruptions:,}")
    print(f"   Disruption rate: {disruption_rate:.2f}%")

    # Severity distribution
    severity_dist = disruption_data[disruption_data['disruption_label'] == 1]['disruption_severity'].value_counts()
    print(f"\n🚨 Disruption Severity Distribution:")
    for severity, count in severity_dist.items():
        print(f"   {severity}: {count} ({count/total_disruptions*100:.1f}%)")

    # Temporal patterns
    disruption_subset = disruption_data[disruption_data['disruption_label'] == 1].copy()
    if len(disruption_subset) > 0:
        disruption_subset['month'] = pd.to_datetime(disruption_subset['date']).dt.month
        disruption_subset['day_of_week'] = pd.to_datetime(disruption_subset['date']).dt.dayofweek

        print(f"\n📅 Temporal Patterns:")
        monthly_dist = disruption_subset['month'].value_counts().sort_index()
        print(f"   Most disrupted months: {monthly_dist.head(3).to_dict()}")

        dow_dist = disruption_subset['day_of_week'].value_counts().sort_index()
        dow_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
        dow_named = {dow_names[k]: v for k, v in dow_dist.items()}
        print(f"   Most disrupted days: {dow_named}")

    # Store/Category analysis
    if len(disruption_subset) > 0:
        store_disruptions = disruption_subset.groupby('store_id')['disruption_label'].count().sort_values(ascending=False)
        cat_disruptions = disruption_subset.groupby('cat_id')['disruption_label'].count().sort_values(ascending=False)

        print(f"\n🏪 Most Affected Stores: {store_disruptions.head(3).to_dict()}")
        print(f"📦 Most Affected Categories: {cat_disruptions.head(3).to_dict()}")

    # Create visualizations
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Severity distribution
    if len(severity_dist) > 0:
        severity_dist.plot(kind='bar', ax=axes[0,0], color='coral')
        axes[0,0].set_title('Disruption Severity Distribution')
        axes[0,0].set_xlabel('Severity Level')
        axes[0,0].set_ylabel('Count')
        axes[0,0].tick_params(axis='x', rotation=45)

    # Monthly pattern
    if len(disruption_subset) > 0:
        monthly_dist.plot(kind='bar', ax=axes[0,1], color='skyblue')
        axes[0,1].set_title('Disruptions by Month')
        axes[0,1].set_xlabel('Month')
        axes[0,1].set_ylabel('Count')

    # Day of week pattern
    if len(disruption_subset) > 0:
        dow_series = pd.Series(dow_named)
        dow_series.plot(kind='bar', ax=axes[1,0], color='lightgreen')
        axes[1,0].set_title('Disruptions by Day of Week')
        axes[1,0].set_xlabel('Day')
        axes[1,0].set_ylabel('Count')
        axes[1,0].tick_params(axis='x', rotation=45)

    # Store distribution (top 10)
    if len(disruption_subset) > 0:
        store_disruptions.head(10).plot(kind='bar', ax=axes[1,1], color='plum')
        axes[1,1].set_title('Top 10 Stores by Disruption Count')
        axes[1,1].set_xlabel('Store ID')
        axes[1,1].set_ylabel('Count')
        axes[1,1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

# Generate visualizations and analysis
if 'disruption_data' in locals():
    visualize_disruption_examples(disruption_data, n_examples=2)
    analyze_disruption_patterns(disruption_data)

# ============================================================================
# SECTION 6: MODEL PREPARATION
# ============================================================================

def prepare_model_features(disruption_data):
    """Prepare features for machine learning models"""

    print("🛠️ Preparing features for modeling...")

    # Select relevant features for modeling
    feature_columns = [
        # Basic identifiers
        'store_id', 'cat_id', 'dept_id',

        # Rolling statistics
        'rolling_mean_7', 'rolling_std_7', 'rolling_cv_7',
        'rolling_mean_14', 'rolling_std_14', 'rolling_cv_14',
        'rolling_mean_28', 'rolling_std_28', 'rolling_cv_28',

        # Z-scores (key anomaly indicators)
        'zscore_7', 'zscore_14', 'zscore_28',

        # Trend features
        'trend_7', 'trend_14', 'trend_28',

        # Calendar features
        'day_of_week', 'month', 'quarter', 'is_weekend',
        'has_event', 'event_count', 'snap_total',

        # Sales features
        'sales'
    ]

    # Filter to available columns
    available_features = [col for col in feature_columns if col in disruption_data.columns